# Конспект к задаче 3

## 1. Постановка задачи

Решается задача Коши для **системы ОДУ**

$$
y'(t)=f(t,y), \qquad y(t_0)=y_0,
$$

где

$$
y(t)\in\mathbb{R}^d.
$$

То есть теперь неизвестная функция может иметь несколько компонент:

$$
y(t)=
\begin{pmatrix}
y_1(t)\\
\dots\\
y_d(t)
\end{pmatrix}.
$$

В текущей реализации в `main.py`:

- при `d=1` остается скалярный пример из задачи 2;
- при `d=2` по умолчанию используется гармонический осциллятор

$$
\begin{cases}
x' = v,\\
v' = -x,
\end{cases}
\qquad
x(0)=1,\; v(0)=0.
$$

Точное решение:

$$
x(t)=\cos t,\qquad v(t)=-\sin t.
$$

## 2. Метод

### 2.1. Формулы метода для системы

Формулы остаются теми же, только теперь $y_n$ и $f_n$ — это векторы.

Предиктор AB4:

$$
y_{n+1}^{(0)} = y_n + \frac{h}{24}
\left(
55 f_n - 59 f_{n-1} + 37 f_{n-2} - 9 f_{n-3}
\right).
$$

Корректор AM4:

$$
y_{n+1}=y_n+\frac{h}{720}
\left(
251f_{n+1}+646f_n-264f_{n-1}+106f_{n-2}-19f_{n-3}
\right),
$$

где

$$
f_k=f(t_k,y_k), \qquad f_{n+1}=f(t_{n+1},y_{n+1}).
$$

Все операции выполняются покомпонентно, но удобно думать о них как о векторных формулах.

### 2.2. Fixed-point итерации

Неявность снимается так же, как в скалярном случае:

$$
y^{(m+1)} = y_n + \frac{h}{720}
\left(
251 f(t_{n+1},y^{(m)}) + 646f_n -264f_{n-1}+106f_{n-2}-19f_{n-3}
\right).
$$

Разница только в критерии остановки: теперь используется бесконечная норма вектора

$$
\|y^{(m+1)}-y^{(m)}\|_\infty < \text{tol}.
$$

Это означает, что контролируется максимальное по модулю изменение среди всех компонент.

### 2.3. Старт метода

Как и в задаче 2, метод не самостартующий. Нужны значения

$$
y_0,\; y_1,\; y_2,\; y_3.
$$

Они получаются следующим образом: $y_0$ задано, а $y_1,y_2,y_3$ вычисляются RK4. Это работает и для системы, потому что RK4 естественно обобщается на векторный случай.

## 3. Точность и сходимость

### 3.1. Порядок метода

Для системы порядок метода не меняется:

- локальная погрешность: $O(h^6)$;
- глобальная погрешность: $O(h^5)$.

Это верно для каждой компоненты и для векторной ошибки в любой эквивалентной норме, в том числе в норме $\|\cdot\|_\infty$.

Если шаг уменьшить в 2 раза, то в асимптотическом режиме ошибка должна уменьшаться примерно в

$$
2^5=32
$$

раз.

### 3.2. Что нужно для сходимости

Как и для скалярного случая:

- правая часть должна быть достаточно гладкой;
- метод должен быть согласованным и нуль-устойчивым;
- шаг должен быть достаточно мал.

Для fixed-point итераций желательно, чтобы отображение было сжимающим. Грубо условие можно оценить как

$$
\frac{251}{720}hL < 1,
$$

где $L$ — константа Липшица по $y$.

Если это не выполняется, итерации сходятся медленно или могут не сойтись вообще. Поэтому метод в такой реализации лучше работает на **нежестких** системах.

### 3.3. Ограничения метода

1. Метод многошаговый, то есть требует стартовых значений.
2. Шаг в реализации фиксированный.
3. Для жестких систем простые итерации могут оказаться недостаточно устойчивыми.
4. Если точное решение неизвестно, то внешний контроль ошибки возможен только по сеточному сравнению, а не напрямую.

## 4. Как считается ошибка в задаче 3

Главное отличие от задачи 2 — ошибка теперь скаляризуется через норму:

$$
\delta(t_i)=\|y_{\text{num}}(t_i)-y_{\text{exact}}(t_i)\|_\infty.
$$

То есть в каждом узле берется максимум по компонентам:

$$
\delta(t_i)=\max_k |y_k^{num}(t_i)-y_k^{exact}(t_i)|.
$$

Дальше по этой величине считаются:

- средняя ошибка:
  $$
  \text{mean\_abs\_error}=\frac{1}{N}\sum_{i=0}^{N-1}\delta(t_i),
  $$
- максимальная ошибка:
  $$
  \text{max\_abs\_error}=\max_i \delta(t_i).
  $$

Это удобно, потому что одно число характеризует ошибку всей системы сразу.

## 5. Как устроена программа

### 5.1. Архитектура

Структура почти та же, что и в задаче 2:

- `solver.py` содержит только численное ядро: `rk4_step`, `solve_adams_moulton4`;
- `input_parser.py` читает `in.txt`, причем `y0` может быть скаляром или вектором;
- `csv_io.py` пишет CSV как для скаляра, так и для системы;
- `plotter.py` строит графики;
- `main.py` задает `f`, `exact` и запускает весь сценарий.

### 5.2. Что изменилось относительно задачи 2

1. Внутри solver все величины рассматриваются как массивы shape `(d,)`.
2. История правых частей хранится как последовательность векторных значений $f_{n-3}, f_{n-2}, f_{n-1}, f_n$.
3. Входной `y0` теперь может быть вектором.
4. CSV-файлы для системы содержат несколько компонент.
5. Ошибка сравнения берется по инф-норме вектора.

### 5.3. Форматы выходных файлов

Сохранены те же имена файлов:

- `exact_xy.csv`
- `num_h_*.csv`
- `num_vs_exact_h_*.csv`
- `summary.csv`
- `comparison.png`

Но формат CSV адаптирован.

#### Если $d=1$

Все как раньше:

- `exact_xy.csv`: `x,y`
- `num_vs_exact_h_*.csv`: `x,y_num,y_exact,delta`

#### Если $d>1$

Файлы становятся многоколонными:

- `exact_xy.csv`: `x,y1,y2,...,yd`
- `num_h_*.csv`: `x,y1,y2,...,yd`
- `num_vs_exact_h_*.csv`: `x,y_num1,...,y_numd,y_exact1,...,y_exactd,delta`

Это и есть backward compatibility: скалярный режим не ломается, а системный просто расширяет формат.

### 5.4. Как строится график

В `plotter.py` реализовано:

- для `d=1` — один обычный график, как в задаче 2;
- для `d>1` — на одном рисунке строятся все компоненты: точные и численные.

То есть для осциллятора на одном рисунке видно и $x(t)$, и $v(t)$, и их численные аппроксимации.

## 6. Что говорить преподавателю про различие задач 2 и 3

### Задача 2

Это скалярный случай:

$$
y(t)\in\mathbb{R}.
$$

Там ошибка — просто

$$
|y_{num}-y_{exact}|.
$$

### Задача 3

Это обобщение на систему:

$$
y(t)\in\mathbb{R}^d.
$$

Метод тот же, но:

- все значения стали векторами;
- критерий сходимости тоже векторный;
- ошибка считается через норму;
- CSV и график адаптированы под несколько компонент.

То есть математическое ядро осталось тем же, а обобщение произошло за счет перехода от числа к вектору.

## 7. Короткий план устного рассказа

1. Сначала формулирую задачу Коши для системы ОДУ.
2. Говорю, что использую predictor-corrector: AB4 как предиктор, AM4 как корректор.
3. Объясняю, что метод неявный, поэтому на каждом шаге решаю уравнение методом простых итераций.
4. Отмечаю, что стартовые значения беру RK4.
5. Говорю о точности: локальная $O(h^6)$, глобальная $O(h^5)$, при уменьшении шага в 2 раза ошибка падает примерно в 32 раза.
6. Отдельно подчеркиваю ограничение: метод хорошо работает на гладких не-жестких задачах, для жестких задач fixed-point может не сходиться.
7. Потом перехожу к программе: разбор входа, расчет, сравнение с exact, CSV, график, summary.

## 8. Одной фразой итог

В задаче 3 та же схема AB4 + AM4 аккуратно обобщена со скалярного случая на систему ОДУ: все формулы работают покомпонентно, ошибка считается по инф-норме вектора, а программа сохраняет совместимость со старым скалярным режимом.